# Groq API Setup + Prompt Engineering Basics

*Day 1 — Week 1: LLM Fundamentals*

## Problem Statement
Build a Groq-powered SSIS failure classifier that tags pipeline error notes with category, severity, and action — and flags when it isn't sure.

## My Approach
I began with a zero-shot to see what the model could do on its own, then layered in few-shot examples to keep the confidence scores consistent. It started as a simple, "naive" setup, but I refactored the whole thing once I saw where it actually started to struggle.

## What I Learned
confidence flagging as more than just a feature- it's the foundation of human-in-the-loop systems. Honestly, a pipeline that flags its own uncertainty is always going to be more trustworthy than one that pretends to have all the answers.

## Where I Got Stuck
Initial few-shot examples were wrong — I accidentally taught the model to label SILENT_FAILURE as OTHER. Correct examples matter more than having examples at all.

## What I'd Do Differently
Define a strict output schema (Pydantic model) instead of trusting the LLM to return consistent field names & intervals across runs. The root key kept changing between "results", "logs", "log_entries" & confidence was between 0 to 1 in zero shot , in few shot , it changed to 0 to 100 — that's fragile in production.

In [7]:
# pip install groq
from groq import Groq

from dotenv import load_dotenv
import json
import os

load_dotenv()  # reads .env from current directory

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

MODEL = "llama-3.1-8b-instant"  # Fast, free, good enough for Week 1

failure_notes = [
    "Package failed at 2:14 AM. OLE DB Source on the Orders extract step threw error 0xC0202009. Truncation likely — column width mismatch in staging.",
    "Flat file connection manager couldn't find the file. Path was D:/drops/sales_daily.csv — file didn't arrive from upstream. Classic late feed.",
    "Memory pressure on the server. The foreach loop enumerating 40 child packages ran out of buffer. Happened before when month-end jobs pile up.",
    "Unknown. Package just stopped. No error in the log. DBA checked — SQL Agent job marked succeeded but downstream table has zero rows.",
]

CATEGORIES = ["DATA_QUALITY", "FILE_MISSING", "RESOURCE_EXHAUSTION", "SILENT_FAILURE", "OTHER"]

## My Solution (Naive)
*First attempt — written before reviewing feedback*

In [19]:
# Cell 2 — One reusable function, prompt is the variable
def classify(notes: list, prompt):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a senior DBA specializing in SSIS pipeline triage."},
            {"role": "user", "content": prompt}
        ],
        response_format={"type": "json_object"}
    )
    return response.choices[0].message.content

In [ ]:
# --- Zero-shot prompt (Cell 3) ---
zero_shot_prompt=f'''
Classify notes into categories : {CATEGORIES} and return json object containing note , category , severity , suggested_action & confidence.
Notes : {failure_notes}
'''
results=classify(failure_notes , zero_shot_prompt)
print(f"{results}")


{
  "notes": [
    {
        "note": "Package failed at 2:14 AM. OLE DB Source on the Orders extract step threw error 0xC0202009. Truncation likely — column width mismatch in staging.",
        "category": "DATA_QUALITY",
        "severity": "high",
        "suggested_action": "Inspect the data for column width mismatch between the source and staging tables and update the table schema accordingly.",
        "confidence": 0.8
    },
    {
        "note": "Flat file connection manager couldn't find the file. Path was D:/drops/sales_daily.csv — file didn't arrive from upstream. Classic late feed.",
        "category": "FILE_MISSING",
        "severity": "medium",
        "suggested_action": "Check the file path for any spelling errors. Ensure that the file arrived from upstream as expected or adjust the package to wait for the file to be uploaded.",
        "confidence": 0.9
    },
    {
        "note": "Memory pressure on the server. The foreach loop enumerating 40 child packages ran out

In [ ]:

# --- Few-shot prompt---

few_shot_prompt=f'''Classify notes into categories : {CATEGORIES} and return json object containing note , category , severity , suggested_action & confidence.
                Example : note : 'Package failed due it column width error' , label : 'Data Quality check'
                , note : 'Unknown. Package just stopped. No error in the log. DBA checked — SQL Agent job marked succeeded but downstream table has zero rows.' , label : 'Other'

                Notes : {failure_notes}
                '''
results=classify(failure_notes ,few_shot_prompt )

print(f"{results}")


{
  "notes": [
       {
           "note": "Package failed at 2:14 AM. OLE DB Source on the Orders extract step threw error 0xC0202009. Truncation likely — column width mismatch in staging.",
           "category": "DATA_QUALITY",
           "severity": "High",
           "suggested_action": "Check staging table column widths and truncate OLE DB source if necessary",
           "confidence": 90
       },
       {
           "note": "Flat file connection manager couldn't find the file. Path was D:/drops/sales_daily.csv — file didn't arrive from upstream. Classic late feed.",
           "category": "FILE_MISSING",
           "severity": "High",
           "suggested_action": "Check file delivery schedule and contact upstream source if necessary",
           "confidence": 90
       },
       {
           "note": "Memory pressure on the server. The foreach loop enumerating 40 child packages ran out of buffer. Happened before when month-end jobs pile up.",
           "category": "RESOURCE_E

In [ ]:

# --- Hard mode: confidence flag ---
few_shot_prompt_2=f'''Classify notes into categories : {CATEGORIES} and return json object containing note , category , severity , suggested_action & confidence (High|Low).
                Example : note : 'Package failed due it column width error' , label : 'Data Quality check'
                , note : 'Unknown. Package just stopped. No error in the log. DBA checked — SQL Agent job marked succeeded but downstream table has zero rows.' , label : 'Other'

                Notes : {failure_notes}
                '''
results=classify(failure_notes ,few_shot_prompt_2 )

print(f"{results}")



{
  "notes": [
      {
         "note": "Package failed at 2:14 AM. OLE DB Source on the Orders extract step threw error 0xC0202009. Truncation likely — column width mismatch in staging.",
         "category": "DATA_QUALITY",
         "severity": "Medium",
         "suggested_action": "Review staging table schema and adjust column widths or data types as needed.",
         "confidence": "High"
      },
      {
         "note": "Flat file connection manager couldn't find the file. Path was D:/drops/sales_daily.csv — file didn't arrive from upstream. Classic late feed.",
         "category": "FILE_MISSING",
         "severity": "High",
         "suggested_action": "Verify upstream file delivery and check for any errors in upstream job.",
         "confidence": "High"
      },
      {
         "note": "Memory pressure on the server. The foreach loop enumerating 40 child packages ran out of buffer. Happened before when month-end jobs pile up.",
         "category": "RESOURCE_EXHAUSTION",
 

## Refactored Solution
The root key kept changing between "results", "logs", "log_entries" , notes .

Confidence interval was also not constant . itwas between 0 to 1 in zero shot , in few shot , it changed to 0 to 100 . 

The function's return needed more refinement .

The way notes was passed in prompt could have been improved .

It would have been more efficient to compare all 3 prompts at last and look which was better rather than manual .

Confidence flag is better then Confidence percentage or number.

In [23]:
# Cell 2 — One reusable function, prompt is the variable
def classify(notes: list, prompt_builder) -> list:
    prompt = prompt_builder(notes)
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a senior DBA specializing in SSIS pipeline triage."},
            {"role": "user", "content": prompt}
        ],
        response_format={"type": "json_object"}
    )
    raw = json.loads(response.choices[0].message.content)
    # Normalize: return whatever list is inside the response
    return next(v for v in raw.values() if isinstance(v, list))

In [24]:
# Cell 3 — Zero-shot prompt
def zero_shot_prompt(notes):
    return f"""Classify each SSIS failure note into exactly one category from {CATEGORIES}.

Return a JSON object with key "results" containing a list. Each item must have:
- note (string)
- category (one of the categories above)
- severity (HIGH | MEDIUM | LOW)
- suggested_action (one imperative sentence)

Notes:
{json.dumps(notes, indent=2)}"""

zero_shot_results = classify(failure_notes, zero_shot_prompt)
print("ZERO-SHOT RESULTS")
print(json.dumps(zero_shot_results, indent=2))

ZERO-SHOT RESULTS
[
  {
    "note": "Package failed at 2:14 AM. OLE DB Source on the Orders extract step threw error 0xC0202009. Truncation likely \u2014 column width mismatch in staging.",
    "category": "DATA_QUALITY",
    "severity": "MEDIUM",
    "suggested_action": "Update the staging table schema to accommodate larger data values."
  },
  {
    "note": "Flat file connection manager couldn't find the file. Path was D:/drops/sales_daily.csv \u2014 file didn't arrive from upstream. Classic late feed.",
    "category": "FILE_MISSING",
    "severity": "LOW",
    "suggested_action": "Verify the file transfer process from upstream to ensure timely delivery of the sales_daily.csv file."
  },
  {
    "note": "Memory pressure on the server. The foreach loop enumerating 40 child packages ran out of buffer. Happened before when month-end jobs pile up.",
    "category": "RESOURCE_EXHAUSTION",
    "severity": "HIGH",
    "suggested_action": "Adjust the server resources or optimize the SSIS pa

In [25]:
# Cell 4 — Few-shot prompt (examples must be correct)
def few_shot_prompt(notes):
    return f"""Classify each SSIS failure note into exactly one category from {CATEGORIES}.

Examples of correct classification:
- "Job failed — column width mismatch caused row truncation in staging table." → DATA_QUALITY, HIGH
- "File not found at expected path. Upstream vendor did not deliver." → FILE_MISSING, HIGH
- "Package completed with status SUCCESS but target table has 0 rows. No error raised." → SILENT_FAILURE, HIGH

Return a JSON object with key "results" containing a list. Each item must have:
- note, category, severity, suggested_action, confidence (HIGH | LOW)

Set confidence to LOW when the note is ambiguous or lacks clear error signals.

Notes:
{json.dumps(notes, indent=2)}"""

few_shot_results = classify(failure_notes, few_shot_prompt)
print("FEW-SHOT RESULTS")
print(json.dumps(few_shot_results, indent=2))

FEW-SHOT RESULTS
[
  {
    "note": "Package failed at 2:14 AM. OLE DB Source on the Orders extract step threw error 0xC0202009. Truncation likely \u2014 column width mismatch in staging.",
    "category": "DATA_QUALITY",
    "severity": "HIGH",
    "suggested_action": "Re-examine column widths in staging table",
    "confidence": "HIGH"
  },
  {
    "note": "Flat file connection manager couldn't find the file. Path was D:/drops/sales_daily.csv \u2014 file didn't arrive from upstream. Classic late feed.",
    "category": "FILE_MISSING",
    "severity": "HIGH",
    "suggested_action": "Check with upstream vendor for file delivery schedule",
    "confidence": "HIGH"
  },
  {
    "note": "Memory pressure on the server. The foreach loop enumerating 40 child packages ran out of buffer. Happened before when month-end jobs pile up.",
    "category": "RESOURCE_EXHAUSTION",
    "severity": "HIGH",
    "suggested_action": "Review and optimize memory usage for month-end jobs",
    "confidence": "H

In [26]:
# Cell 5 — Side-by-side comparison (this is the actual deliverable)
print(f"{'NOTE':<30} {'ZERO-SHOT CAT':<22} {'FEW-SHOT CAT':<22} {'CONFIDENCE'}")
print("-" * 90)

for zs, fs in zip(zero_shot_results, few_shot_results):
    note_preview = zs["note"][:28] + ".."
    changed = " ← CHANGED" if zs["category"] != fs["category"] else ""
    confidence = fs.get("confidence", "N/A")
    low_flag = " ⚠ REVIEW" if confidence == "LOW" else ""
    print(f"{note_preview:<30} {zs['category']:<22} {fs['category']:<22} {confidence}{low_flag}{changed}")

NOTE                           ZERO-SHOT CAT          FEW-SHOT CAT           CONFIDENCE
------------------------------------------------------------------------------------------
Package failed at 2:14 AM. O.. DATA_QUALITY           DATA_QUALITY           HIGH
Flat file connection manager.. FILE_MISSING           FILE_MISSING           HIGH
Memory pressure on the serve.. RESOURCE_EXHAUSTION    RESOURCE_EXHAUSTION    HIGH
Unknown. Package just stoppe.. SILENT_FAILURE         SILENT_FAILURE         LOW ⚠ REVIEW
